In [3]:
import os
import struct
import random
from tqdm import tqdm
import numpy as np
from tonic.datasets.hsd import SSC  # SSC 클래스 임포트

# ───────── USER PARAMETERS ───────────────────────────────────────────────────
DOWNLOAD_DIR    = "/home/sungminlee/speakmin/run"   # contains ssc_train.h5 / ssc_valid.h5 / ssc_test.h5
TRAIN_OUT_DIR   = "./ssc_train"                    # train0.bin…train9.bin
VALID_OUT_DIR   = "./ssc_valid"                    # valid0.bin
TEST_OUT_DIR    = "./ssc_test"                     # test0.bin
TOTAL_CHANNELS  = 700
TARGET_CHANNELS = 64
TRAIN_PARTS     = 10
RNG_SEED        = 42
# ─────────────────────────────────────────────────────────────────────────────

def group_neuron_indices(neuron_idx: np.ndarray,
                         total: int,
                         target: int) -> np.ndarray:
    gs = total // target
    rem = total % target
    mapped = neuron_idx // gs
    if rem:
        mask = neuron_idx >= total - rem
        mapped[mask] = target - 1
    return mapped.astype(np.uint16)

def write_sample(fh, label, data_index, global_id, times, neurons):
    n = len(times)
    # header: >BHIIBB
    hdr = struct.pack(">BHIIBB", int(label), int(data_index),
                      int(global_id), int(n), 0, 0)
    fh.write(hdr)
    # interleaved payload: I H repeated
    payload = np.column_stack((times, neurons)).ravel()
    fmt = ">" + "IH"*n
    fh.write(struct.pack(fmt, *payload))

def convert_split(ds, out_dir, prefix, parts, shuffle, rng):
    os.makedirs(out_dir, exist_ok=True)
    handles = {i: open(f"{out_dir}/{prefix}{i}.bin", "wb")
               for i in range(parts)}
    per_label_count = {}
    gid = 0
    idxs = list(range(len(ds)))
    if shuffle: rng.shuffle(idxs)

    for k, i in enumerate(tqdm(idxs, desc=f"Packing SSC-{prefix}")):
        events, label = ds[i]
        times   = events['t'].astype(np.uint32)
        neurons = group_neuron_indices(events['x'].astype(np.uint32),
                                       TOTAL_CHANNELS,
                                       TARGET_CHANNELS)
        data_index = per_label_count.get(label, 0)
        fid = k % parts
        write_sample(handles[fid],
                     label, data_index, gid,
                     times, neurons)
        per_label_count[label] = data_index + 1
        gid += 1

    for fh in handles.values(): fh.close()
    counts = [cnt for cnt in per_label_count.values()]
    print(f"SSC-{prefix}: generated {parts} files, label counts (per-label total samples): {per_label_count}")

if __name__ == "__main__":
    rng = random.Random(RNG_SEED)
    # 학습·유효·테스트 데이터셋 로드
    train_ds = SSC(save_to=DOWNLOAD_DIR, split="train")
    valid_ds = SSC(save_to=DOWNLOAD_DIR, split="valid")
    test_ds  = SSC(save_to=DOWNLOAD_DIR, split="test")

    # 학습 셔플 → 10 shards
    convert_split(train_ds, TRAIN_OUT_DIR, "train", parts=TRAIN_PARTS, shuffle=True,  rng=rng)
    # valid/test는 하나의 파일로
    convert_split(valid_ds, VALID_OUT_DIR, "valid", parts=1, shuffle=False, rng=rng)
    convert_split(test_ds,  TEST_OUT_DIR,  "test",  parts=1, shuffle=False, rng=rng)


  0%|          | 0/1189044141 [00:00<?, ?it/s]

Extracting /home/sungminlee/speakmin/run/SSC/ssc_train.h5.zip to /home/sungminlee/speakmin/run/SSC


  0%|          | 0/155548491 [00:00<?, ?it/s]

Extracting /home/sungminlee/speakmin/run/SSC/ssc_valid.h5.zip to /home/sungminlee/speakmin/run/SSC


  0%|          | 0/323314930 [00:00<?, ?it/s]

...

SSC-train: generated 10 files, label counts (per-label total samples): {10: 2648, 12: 2756, 2: 2779, 21: 2764, 18: 2628, 33: 1463, 19: 1479, 16: 1526, 30: 1242, 14: 2798, 0: 2885, 13: 2860, 8: 2712, 20: 1434, 9: 2816, 25: 2815, 4: 2622, 1: 2802, 34: 2665, 24: 1527, 11: 1123, 28: 1431, 31: 1160, 32: 1213, 27: 2757, 6: 2768, 15: 1518, 26: 1532, 7: 2842, 22: 2690, 5: 2868, 29: 1408, 23: 1156, 3: 2621, 17: 1158}


Packing SSC-valid: 100%|██████████| 9981/9981 [02:09<00:00, 76.94it/s] 


SSC-valid: generated 1 files, label counts (per-label total samples): {0: 384, 1: 351, 2: 345, 3: 356, 4: 373, 5: 367, 6: 378, 7: 387, 8: 346, 9: 356, 10: 373, 11: 146, 12: 363, 13: 397, 14: 377, 15: 195, 16: 195, 17: 139, 18: 350, 19: 182, 20: 204, 21: 350, 22: 352, 23: 132, 24: 193, 25: 406, 26: 197, 27: 372, 28: 219, 29: 213, 30: 159, 31: 128, 32: 153, 33: 180, 34: 363}


Packing SSC-test: 100%|██████████| 20382/20382 [04:40<00:00, 72.66it/s]


SSC-test: generated 1 files, label counts (per-label total samples): {0: 783, 1: 737, 2: 756, 3: 750, 4: 733, 5: 817, 6: 714, 7: 769, 8: 729, 9: 762, 10: 724, 11: 288, 12: 726, 13: 787, 14: 742, 15: 400, 16: 379, 17: 295, 18: 745, 19: 403, 20: 384, 21: 758, 22: 759, 23: 291, 24: 403, 25: 720, 26: 399, 27: 751, 28: 404, 29: 393, 30: 358, 31: 287, 32: 298, 33: 388, 34: 750}


In [ ]:
import os
import struct
import random
from tqdm import tqdm
import numpy as np
from tonic.datasets.hsd import SSC  # SSC 클래스 임포트

# ───────── USER PARAMETERS ───────────────────────────────────────────────────
DOWNLOAD_DIR    = "/home/sungminlee/speakmin/run"   # contains ssc_train.h5 / ssc_valid.h5 / ssc_test.h5
TRAIN_OUT_DIR   = "./SSC_700"                    # train0.bin…train9.bin
VALID_OUT_DIR   = "./SSC_700"                    # valid0.bin
TEST_OUT_DIR    = "./SSC_700"                     # test0.bin
TOTAL_CHANNELS  = 700
TARGET_CHANNELS = 700
TRAIN_PARTS     = 10
RNG_SEED        = 42
# ─────────────────────────────────────────────────────────────────────────────

def group_neuron_indices(neuron_idx: np.ndarray,
                         total: int,
                         target: int) -> np.ndarray:
    gs = total // target
    rem = total % target
    mapped = neuron_idx // gs
    if rem:
        mask = neuron_idx >= total - rem
        mapped[mask] = target - 1
    return mapped.astype(np.uint16)

def write_sample(fh, label, data_index, global_id, times, neurons):
    n = len(times)
    # header: >BHIIBB
    hdr = struct.pack(">BHIIBB", int(label), int(data_index),
                      int(global_id), int(n), 0, 0)
    fh.write(hdr)
    # interleaved payload: I H repeated
    payload = np.column_stack((times, neurons)).ravel()
    fmt = ">" + "IH"*n
    fh.write(struct.pack(fmt, *payload))

def convert_split(ds, out_dir, prefix, parts, shuffle, rng):
    os.makedirs(out_dir, exist_ok=True)
    handles = {i: open(f"{out_dir}/{prefix}{i}.bin", "wb")
               for i in range(parts)}
    per_label_count = {}
    gid = 0
    idxs = list(range(len(ds)))
    if shuffle: rng.shuffle(idxs)

    for k, i in enumerate(tqdm(idxs, desc=f"Packing SSC-{prefix}")):
        events, label = ds[i]
        times   = events['t'].astype(np.uint32)
        neurons = group_neuron_indices(events['x'].astype(np.uint32),
                                       TOTAL_CHANNELS,
                                       TARGET_CHANNELS)
        data_index = per_label_count.get(label, 0)
        fid = k % parts
        write_sample(handles[fid],
                     label, data_index, gid,
                     times, neurons)
        per_label_count[label] = data_index + 1
        gid += 1

    for fh in handles.values(): fh.close()
    counts = [cnt for cnt in per_label_count.values()]
    print(f"SSC-{prefix}: generated {parts} files, label counts (per-label total samples): {per_label_count}")

if __name__ == "__main__":
    rng = random.Random(RNG_SEED)
    # 학습·유효·테스트 데이터셋 로드
    train_ds = SSC(save_to=DOWNLOAD_DIR, split="train")
    valid_ds = SSC(save_to=DOWNLOAD_DIR, split="valid")
    test_ds  = SSC(save_to=DOWNLOAD_DIR, split="test")

    # 학습 셔플 → 10 shards
    convert_split(train_ds, TRAIN_OUT_DIR, "train", parts=TRAIN_PARTS, shuffle=True,  rng=rng)
    # valid/test는 하나의 파일로
    convert_split(valid_ds, VALID_OUT_DIR, "valid", parts=1, shuffle=False, rng=rng)
    convert_split(test_ds,  TEST_OUT_DIR,  "test",  parts=1, shuffle=False, rng=rng)


Packing SSC-train: 100%|██████████| 75466/75466 [24:58<00:00, 50.36it/s]


SSC-train: generated 10 files, label counts (per-label total samples): {10: 2648, 12: 2756, 2: 2779, 21: 2764, 18: 2628, 33: 1463, 19: 1479, 16: 1526, 30: 1242, 14: 2798, 0: 2885, 13: 2860, 8: 2712, 20: 1434, 9: 2816, 25: 2815, 4: 2622, 1: 2802, 34: 2665, 24: 1527, 11: 1123, 28: 1431, 31: 1160, 32: 1213, 27: 2757, 6: 2768, 15: 1518, 26: 1532, 7: 2842, 22: 2690, 5: 2868, 29: 1408, 23: 1156, 3: 2621, 17: 1158}


Packing SSC-valid: 100%|██████████| 9981/9981 [01:50<00:00, 90.73it/s] 


SSC-valid: generated 1 files, label counts (per-label total samples): {0: 384, 1: 351, 2: 345, 3: 356, 4: 373, 5: 367, 6: 378, 7: 387, 8: 346, 9: 356, 10: 373, 11: 146, 12: 363, 13: 397, 14: 377, 15: 195, 16: 195, 17: 139, 18: 350, 19: 182, 20: 204, 21: 350, 22: 352, 23: 132, 24: 193, 25: 406, 26: 197, 27: 372, 28: 219, 29: 213, 30: 159, 31: 128, 32: 153, 33: 180, 34: 363}


Packing SSC-test: 100%|██████████| 20382/20382 [04:01<00:00, 84.48it/s] 


SSC-test: generated 1 files, label counts (per-label total samples): {0: 783, 1: 737, 2: 756, 3: 750, 4: 733, 5: 817, 6: 714, 7: 769, 8: 729, 9: 762, 10: 724, 11: 288, 12: 726, 13: 787, 14: 742, 15: 400, 16: 379, 17: 295, 18: 745, 19: 403, 20: 384, 21: 758, 22: 759, 23: 291, 24: 403, 25: 720, 26: 399, 27: 751, 28: 404, 29: 393, 30: 358, 31: 287, 32: 298, 33: 388, 34: 750}


In [5]:
import os
import struct
import random
from tqdm import tqdm
import numpy as np
from tonic.datasets.hsd import SSC  # SSC 클래스 임포트

# ───────── USER PARAMETERS ───────────────────────────────────────────────────
DOWNLOAD_DIR    = "/home/sungminlee/speakmin/run"   # contains ssc_train.h5 / ssc_valid.h5 / ssc_test.h5
TRAIN_OUT_DIR   = "./SSC_350"                    # train0.bin…train9.bin
VALID_OUT_DIR   = "./SSC_350"                    # valid0.bin
TEST_OUT_DIR    = "./SSC_350"                     # test0.bin
TOTAL_CHANNELS  = 700
TARGET_CHANNELS = 350
TRAIN_PARTS     = 10
RNG_SEED        = 42
# ─────────────────────────────────────────────────────────────────────────────

def group_neuron_indices(neuron_idx: np.ndarray,
                         total: int,
                         target: int) -> np.ndarray:
    gs = total // target
    rem = total % target
    mapped = neuron_idx // gs
    if rem:
        mask = neuron_idx >= total - rem
        mapped[mask] = target - 1
    return mapped.astype(np.uint16)

def write_sample(fh, label, data_index, global_id, times, neurons):
    n = len(times)
    # header: >BHIIBB
    hdr = struct.pack(">BHIIBB", int(label), int(data_index),
                      int(global_id), int(n), 0, 0)
    fh.write(hdr)
    # interleaved payload: I H repeated
    payload = np.column_stack((times, neurons)).ravel()
    fmt = ">" + "IH"*n
    fh.write(struct.pack(fmt, *payload))

def convert_split(ds, out_dir, prefix, parts, shuffle, rng):
    os.makedirs(out_dir, exist_ok=True)
    handles = {i: open(f"{out_dir}/{prefix}{i}.bin", "wb")
               for i in range(parts)}
    per_label_count = {}
    gid = 0
    idxs = list(range(len(ds)))
    if shuffle: rng.shuffle(idxs)

    for k, i in enumerate(tqdm(idxs, desc=f"Packing SSC-{prefix}")):
        events, label = ds[i]
        times   = events['t'].astype(np.uint32)
        neurons = group_neuron_indices(events['x'].astype(np.uint32),
                                       TOTAL_CHANNELS,
                                       TARGET_CHANNELS)
        data_index = per_label_count.get(label, 0)
        fid = k % parts
        write_sample(handles[fid],
                     label, data_index, gid,
                     times, neurons)
        per_label_count[label] = data_index + 1
        gid += 1

    for fh in handles.values(): fh.close()
    counts = [cnt for cnt in per_label_count.values()]
    print(f"SSC-{prefix}: generated {parts} files, label counts (per-label total samples): {per_label_count}")

if __name__ == "__main__":
    rng = random.Random(RNG_SEED)
    # 학습·유효·테스트 데이터셋 로드
    train_ds = SSC(save_to=DOWNLOAD_DIR, split="train")
    valid_ds = SSC(save_to=DOWNLOAD_DIR, split="valid")
    test_ds  = SSC(save_to=DOWNLOAD_DIR, split="test")

    # 학습 셔플 → 10 shards
    convert_split(train_ds, TRAIN_OUT_DIR, "train", parts=TRAIN_PARTS, shuffle=True,  rng=rng)
    # valid/test는 하나의 파일로
    convert_split(valid_ds, VALID_OUT_DIR, "valid", parts=1, shuffle=False, rng=rng)
    convert_split(test_ds,  TEST_OUT_DIR,  "test",  parts=1, shuffle=False, rng=rng)


Packing SSC-train: 100%|██████████| 75466/75466 [18:20<00:00, 68.57it/s] 


SSC-train: generated 10 files, label counts (per-label total samples): {10: 2648, 12: 2756, 2: 2779, 21: 2764, 18: 2628, 33: 1463, 19: 1479, 16: 1526, 30: 1242, 14: 2798, 0: 2885, 13: 2860, 8: 2712, 20: 1434, 9: 2816, 25: 2815, 4: 2622, 1: 2802, 34: 2665, 24: 1527, 11: 1123, 28: 1431, 31: 1160, 32: 1213, 27: 2757, 6: 2768, 15: 1518, 26: 1532, 7: 2842, 22: 2690, 5: 2868, 29: 1408, 23: 1156, 3: 2621, 17: 1158}


Packing SSC-valid: 100%|██████████| 9981/9981 [01:13<00:00, 134.93it/s]


SSC-valid: generated 1 files, label counts (per-label total samples): {0: 384, 1: 351, 2: 345, 3: 356, 4: 373, 5: 367, 6: 378, 7: 387, 8: 346, 9: 356, 10: 373, 11: 146, 12: 363, 13: 397, 14: 377, 15: 195, 16: 195, 17: 139, 18: 350, 19: 182, 20: 204, 21: 350, 22: 352, 23: 132, 24: 193, 25: 406, 26: 197, 27: 372, 28: 219, 29: 213, 30: 159, 31: 128, 32: 153, 33: 180, 34: 363}


Packing SSC-test: 100%|██████████| 20382/20382 [03:15<00:00, 104.09it/s]

SSC-test: generated 1 files, label counts (per-label total samples): {0: 783, 1: 737, 2: 756, 3: 750, 4: 733, 5: 817, 6: 714, 7: 769, 8: 729, 9: 762, 10: 724, 11: 288, 12: 726, 13: 787, 14: 742, 15: 400, 16: 379, 17: 295, 18: 745, 19: 403, 20: 384, 21: 758, 22: 759, 23: 291, 24: 403, 25: 720, 26: 399, 27: 751, 28: 404, 29: 393, 30: 358, 31: 287, 32: 298, 33: 388, 34: 750}
